In [32]:
import pandas as pd
import geopandas as gpd
import numpy as np
import ast
import optuna
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.feature_selection import RFE
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor, StackingRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split, GridSearchCV, GroupShuffleSplit, RandomizedSearchCV, cross_val_score, KFold
from shapely.geometry import LineString, Point
from tqdm import tqdm
from itertools import combinations
from sklearn.neighbors import BallTree
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler, FunctionTransformer, PolynomialFeatures, RobustScaler
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.impute import SimpleImputer
from sklearn.ensemble import GradientBoostingRegressor
from xgboost import XGBRegressor, plot_importance, DMatrix, train
from lightgbm import LGBMRegressor
import shap
from scipy.stats import zscore
from scipy.spatial import cKDTree
import os
import fiona
import random
from geopy.distance import geodesic

In [33]:
doubs_data = gpd.read_file("../datasets_par_departement/departement-25-doubs-original.geojson")
caserne = gpd.read_file("../data/caserne/2024_06_21_CIS.shp")
compagnies = gpd.read_file("../data/caserne/2024_06_21_Compagnies.shp")
firepoint_1 = gpd.read_file("../data/hexagones_firepoint_1.geojson")
firepoint_2 = gpd.read_file("../data/hexagones_firepoint_2.geojson")
firepoint_3 = gpd.read_file("../data/hexagones_firepoint_3.geojson")

In [1]:
def extract_vertices(polygon):
    return list(polygon.exterior.coords)

def find_nearest_neighbors(target_hex, all_hexagons, max_distance=300000) :
    target_vertices = extract_vertices(target_hex.geometry)
    hex_distances = {}

    for idx, row in all_hexagons.iterrows():
        if row.hex_id == target_hex.hex_id:
            continue

        train_vertices = extract_vertices(row.geometry)

        min_distance = float("inf")
        for v1 in target_vertices:
            for v2 in train_vertices:
                d = Point(v1).distance(Point(v2))
                if d < min_distance:
                    min_distance = d

        distance_km = min_distance * 111
        if distance_km <= max_distance:
            hex_distances[row.hex_id] = distance_km

    nearest_neighbors = sorted(hex_distances.items(), key=lambda x: x[1])[:6]
    return [hex_id for hex_id, _ in nearest_neighbors]